# 🔀 Hybrid Search RAG: Combining BM25 and Vector Search

**Reference:** [Hybrid Search RAG GitHub](https://github.com/siddharth-Kharche/Hybrid-search-RAG/tree/main)

---

## 📚 Learning Objectives

By the end of this notebook, you will understand:
1. How to implement **Hybrid Search** combining BM25 (keyword) and Vector (semantic) search
2. How to use **EnsembleRetriever** to merge multiple retrieval strategies
3. How to build a complete RAG pipeline with hybrid retrieval
4. How to explain the benefits of hybrid search for different query types

---

## 🧠 Why Hybrid Search?

| Query Type | Best Retriever | Example |
|------------|----------------|---------|
| Exact keyword | BM25 | "Python 3.12 release notes" |
| Semantic meaning | Vector | "How to make code run faster" |
| Mixed queries | **Hybrid** | "Python optimization techniques" |

Hybrid search gets the **best of both worlds**!

---

<a href="https://colab.research.google.com/drive/1R6fYYtk_kTUrFj_WWCskLl954BgZHkQP?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

### Hybrid (Ensamble) Search

Hybrid Search is a technique that combines multiple search methods to improve retrieval performance. Typically, it combines traditional keyword-based search (like BM25) with semantic search (using embeddings). This approach can provide better results than either method alone, especially for queries that have both keyword-specific and semantic aspects.

### Hybrid-search RAG Implementation:

1. **Hybrid Retrieval:** We use the EnsembleRetriever to get relevant documents using both keyword-based and semantic search.
2. **Response Generation:** Using the retrieved context, we generate a final response to the original query.
3. **Hybrid Search Explanation:** We generate an explanation of how the hybrid search process might have improved the retrieval of relevant information.

# Setup

1. **[LLM](https://groq.com/):** Groq's free Open source LLM endpoints([Groq API Key](https://console.groq.com/keys))
2. **[Vector Store](https://www.pinecone.io/learn/vector-database/):** [ChromaDB](https://www.trychroma.com/)
3. **[Embedding Model](https://qdrant.tech/articles/what-are-embeddings/):** [nomic-embed-text-v1.5](https://www.nomic.ai/blog/posts/nomic-embed-text-v1)
4. **[LLM Framework](https://python.langchain.com/v0.2/docs/introduction/):** LangChain
5. **[Huggingface API Key](https://huggingface.co/settings/tokens)**

## 📦 Step 1: Install Required Libraries

The following packages are needed:
- **langchain / langchain-classic**: core framework + `EnsembleRetriever`
- **langchain-groq**: Groq LLM integration
- **langchain-chroma**: ChromaDB vector store
- **langchain-community**: `WebBaseLoader`, `BM25Retriever`
- **langchain-openai**: embeddings for the semantic half of hybrid search
- **rank_bm25**: BM25 retrieval algorithm
- **python-dotenv**: loads API keys from the repo-root `.env`

In [ ]:
# Already installed in this repo's environment — uncomment only if running elsewhere (e.g. Colab).
# !pip install -q -U \
#      langchain \
#      langchain-classic \
#      langchain-groq \
#      langchain-chroma \
#      langchain-community \
#      langchain-openai \
#      rank_bm25 \
#      python-dotenv

## 📦 Step 2: Import Required Libraries

Import all necessary components for building the hybrid search RAG pipeline.

In [ ]:
# ============================================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================================

# ChatGroq: Interface to Groq's fast LLM inference API
from langchain_groq import ChatGroq

# OpenAIEmbeddings: embedding model used for the semantic half of hybrid search
from langchain_openai import OpenAIEmbeddings

# Chroma: Open-source vector database for storing embeddings
from langchain_chroma import Chroma

# RecursiveCharacterTextSplitter: Smart text chunking that preserves context
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Prompt templates for structuring LLM inputs
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

# WebBaseLoader: Loads content from web URLs
from langchain_community.document_loaders import WebBaseLoader

# BM25Retriever: Keyword-based retrieval using BM25 algorithm
# EnsembleRetriever: Combines multiple retrievers with weighted scoring
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

print("✅ All libraries imported successfully!")

In [ ]:
import os

from dotenv import load_dotenv

# Loads GROQ_API_KEY (LLM) and OPENAI_API_KEY (embeddings) from the repo-root .env.
# override=True so a stale/empty value already in the kernel can't win.
load_dotenv(override=True)

#### API keys

Both keys are read from the `.env` at the repo root — no interactive prompt needed:

- **`GROQ_API_KEY`** — free open-source LLM inference ([create one here](https://console.groq.com/keys))
- **`OPENAI_API_KEY`** — embeddings for the semantic half of hybrid search

In [ ]:
for key in ("GROQ_API_KEY", "OPENAI_API_KEY"):
    if not os.environ.get(key):
        raise ValueError(f"{key} is not set — add it to the .env at the repo root")

print("✅ GROQ_API_KEY and OPENAI_API_KEY loaded")

## 🔧 Step 4: Define Data Loading Function

This function loads content from a web URL and splits it into manageable chunks for retrieval.

In [ ]:
def load_and_process_data(url):
    """
    Load content from a web URL and split into chunks for retrieval.
    
    Args:
        url (str): The URL to load content from
        
    Returns:
        list: List of Document chunks ready for indexing
        
    💡 Tips:
        - Adjust chunk_size based on your embedding model's context window
        - chunk_overlap helps preserve context at chunk boundaries
        - Experiment with different values for optimal results
    """
    # -------------------------------------------------------------------------
    # Step 1: Load data from web URL
    # WebBaseLoader extracts text content from web pages
    # -------------------------------------------------------------------------
    loader = WebBaseLoader(url)
    data = loader.load()
    print(f"📄 Loaded {len(data)} document(s) from URL")

    # -------------------------------------------------------------------------
    # Step 2: Split text into chunks
    # Parameters:
    # - chunk_size=500: Maximum characters per chunk
    # - chunk_overlap=50: Characters shared between adjacent chunks
    # -------------------------------------------------------------------------
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,      # Keep chunks small for precise retrieval
        chunk_overlap=50     # Overlap preserves context between chunks
    )
    chunks = text_splitter.split_documents(data)
    print(f"✅ Split into {len(chunks)} chunks")

    return chunks

## 🔧 Step 5: Create Hybrid Retriever (BM25 + Vector)

This is the core of hybrid search - combining keyword-based BM25 with semantic vector search using EnsembleRetriever.

In [ ]:
def create_retrievers(chunks):
    """
    Create a hybrid retriever combining BM25 and vector search.
    
    Args:
        chunks (list): List of Document chunks to index
        
    Returns:
        EnsembleRetriever: Hybrid retriever combining both search methods
        
    🔑 Key Components:
        1. BM25 Retriever: Keyword-based, excels at exact matches
        2. Vector Retriever: Semantic search, understands meaning
        3. EnsembleRetriever: Merges results with configurable weights
    """
    # -------------------------------------------------------------------------
    # Step 1: Create embedding model for vector search
    # Using OpenAI text-embedding-3-small - 1536 dimensions, no local GPU/torch
    # needed. (The original notebook used nomic-embed-text-v1.5 locally via
    # HuggingFaceEmbeddings, which requires the ~2.5 GB torch stack.)
    # -------------------------------------------------------------------------
    print("🔄 Loading embedding model...")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    
    # -------------------------------------------------------------------------
    # Step 2: Create vector store (ChromaDB)
    # ChromaDB stores embeddings for efficient similarity search
    # -------------------------------------------------------------------------
    print("📊 Creating vector store...")
    vectorstore = Chroma.from_documents(chunks, embeddings)
    
    # -------------------------------------------------------------------------
    # Step 3: Create BM25 retriever (keyword-based)
    # BM25 (Best Matching 25) ranks documents using term frequency
    # Great for exact keyword matches and technical terms
    # -------------------------------------------------------------------------
    print("🔤 Creating BM25 retriever...")
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = 5  # Retrieve top 5 documents

    # -------------------------------------------------------------------------
    # Step 4: Combine into EnsembleRetriever (HYBRID SEARCH)
    # weights=[0.5, 0.5] means equal importance to both methods
    # Adjust weights based on your use case:
    #   - More keyword-heavy queries: [0.7, 0.3]
    #   - More semantic queries: [0.3, 0.7]
    #   - Balanced (default): [0.5, 0.5]
    # -------------------------------------------------------------------------
    print("🔀 Creating hybrid (ensemble) retriever...")
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vectorstore.as_retriever(search_kwargs={"k": 5})],
        weights=[0.5, 0.5]  # Equal weight to keyword and semantic search
    )
    
    print("✅ Hybrid retriever created!")
    return ensemble_retriever

## 🛡️ Step 6: Helper Function for Safe LLM Calls

A utility function that handles LLM invocations with proper error handling.

In [ ]:
def safe_llm_call(prompt, **kwargs):
    """
    Safely invoke the LLM with error handling.
    
    Args:
        prompt: PromptTemplate to format and send to LLM
        **kwargs: Variables to substitute in the prompt template
        
    Returns:
        str: LLM response content or error message
        
    💡 Best Practice:
        Always wrap LLM calls in try-except blocks to handle:
        - API rate limits
        - Network timeouts
        - Invalid responses
        - Token limit exceeded
    """
    try:
        # Format the prompt with provided variables and invoke LLM
        response = llm.invoke(prompt.format(**kwargs))
        return response.content if response else "No response generated."
    except Exception as e:
        # Log the error and return a user-friendly message
        print(f"❌ Error in LLM call: {e}")
        return "An error occurred while generating the response."

## 🚀 Step 7: Hybrid Search RAG Function

This is the main RAG function that:
1. **Hybrid Retrieval:** Uses EnsembleRetriever to get relevant documents using both keyword-based and semantic search
2. **Response Generation:** Uses the retrieved context to generate a final response
3. **Hybrid Search Explanation:** Generates an explanation of how hybrid search improved retrieval

### Pipeline Flow:
```
Query → Hybrid Retriever → Retrieved Docs → LLM → Answer + Explanation
```

In [ ]:
def hybrid_search_rag(query, ensemble_retriever, llm):
    """
    Execute a complete hybrid search RAG pipeline.
    
    Args:
        query (str): User's question
        ensemble_retriever: Hybrid retriever combining BM25 + Vector search
        llm: Language model for generating responses
        
    Returns:
        dict: Contains query, final_answer, hybrid_search_explanation, and retrieved_context
        
    🔑 Pipeline Steps:
        1. Retrieve documents using hybrid search (BM25 + Vector)
        2. Generate answer using retrieved context
        3. Explain how hybrid search helped (for educational purposes)
    """
    print(f"🔍 Processing query: \"{query}\"")
    
    # -------------------------------------------------------------------------
    # Step 1: Hybrid Retrieval
    # The EnsembleRetriever combines results from BM25 and vector search
    # This gives us the best of both keyword matching and semantic understanding
    # -------------------------------------------------------------------------
    print("📥 Retrieving documents using hybrid search...")
    retrieved_docs = ensemble_retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    print(f"   Retrieved {len(retrieved_docs)} documents")

    # -------------------------------------------------------------------------
    # Step 2: Generate Response
    # Use the retrieved context to answer the user's question
    # The prompt instructs the LLM to use context when relevant
    # -------------------------------------------------------------------------
    print("🤖 Generating response...")
    response_prompt = PromptTemplate.from_template(
        "You are an AI assistant tasked with answering questions based on the provided context. "
        "The context contains information retrieved using a hybrid search method combining keyword-based and semantic search. "
        "Please provide a comprehensive answer to the question, using the context when relevant "
        "and your general knowledge when necessary.\n\n"
        "Context:\n{context}\n\n"
        "Question: {query}\n"
        "Answer:"
    )
    final_answer = safe_llm_call(response_prompt, context=context, query=query)

    # -------------------------------------------------------------------------
    # Step 3: Generate Hybrid Search Explanation (Educational)
    # This explains how the hybrid approach benefited the retrieval
    # Useful for understanding when to use hybrid search
    # -------------------------------------------------------------------------
    print("📝 Generating hybrid search explanation...")
    explanation_prompt = PromptTemplate.from_template(
        "Explain how the hybrid search process, combining keyword-based and semantic search, "
        "might have improved the retrieval of relevant information for answering the given query. "
        "Consider the potential benefits of this approach compared to using only one search method.\n\n"
        "Query: {query}\n"
        "Explanation:"
    )
    hybrid_search_explanation = safe_llm_call(explanation_prompt, query=query)

    print("✅ RAG pipeline complete!")
    
    return {
        "query": query,
        "final_answer": final_answer,
        "hybrid_search_explanation": hybrid_search_explanation,
        "retrieved_context": context
    }

## 🏃 Step 8: Initialize LLM and Load Data

Now we'll:
1. Initialize the Groq LLM (Llama 3, 8B parameters)
2. Load and process data from Wikipedia
3. Create the hybrid retriever with our processed chunks

In [ ]:
# ============================================================================
# INITIALIZE LLM
# ============================================================================
# Using Groq for fast, free inference of Llama 3
# - model: llama3-8b-8192 (8B parameters, 8192 context window)
# - temperature: 0.5 (balanced creativity/consistency)

print("🤖 Initializing LLM...")
llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0.5  # Moderate randomness for balanced responses
)
print("✅ LLM initialized: Llama 3 8B via Groq")

# ============================================================================
# LOAD AND PROCESS DATA
# ============================================================================
# Loading Wikipedia article about AI as our knowledge base
# This provides a good test case with both technical terms and concepts

print("\n📥 Loading data from Wikipedia...")
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
chunks = load_and_process_data(url)

# ============================================================================
# CREATE HYBRID RETRIEVER
# ============================================================================
# This combines BM25 (keyword) and Vector (semantic) search

print("\n🔀 Creating hybrid retriever...")
ensemble_retriever = create_retrievers(chunks)
print("\n" + "=" * 60)
print("🎉 SETUP COMPLETE! Ready for hybrid search queries.")
print("=" * 60)

## 🧪 Step 9: Run Hybrid Search RAG with Example Queries

Now let's test our hybrid search RAG pipeline with several example queries about artificial intelligence.

### What We'll See:
1. **Final Answer**: Generated using hybrid-retrieved context
2. **Hybrid Search Explanation**: How combining methods helped
3. **Retrieved Context**: The actual documents retrieved (first 300 chars)

### Key Implementation Points:
- **BM25**: Catches exact keyword matches (e.g., "machine learning", "neural networks")
- **Vector Search**: Captures semantic meaning (e.g., "AI ethics" → finds moral implications)
- **Combined**: Gets the best of both for comprehensive answers

In [ ]:
# ============================================================================
# EXAMPLE QUERIES
# ============================================================================
# These queries test different aspects of hybrid search:
# 1. Healthcare applications - Tests domain-specific keyword + semantic retrieval
# 2. Machine learning concept - Tests technical term matching + conceptual understanding
# 3. Ethical implications - Tests abstract concept retrieval

queries = [
    "What are the main applications of artificial intelligence in healthcare?",
    "Explain the concept of machine learning and its relationship to AI.",
    "Discuss the ethical implications of AI in decision-making processes."
]

# ============================================================================
# RUN HYBRID SEARCH RAG FOR EACH QUERY
# ============================================================================

for i, query in enumerate(queries, 1):
    print("\n" + "=" * 70)
    print(f"🔍 QUERY {i}: {query}")
    print("=" * 70)
    
    # Execute the hybrid search RAG pipeline
    result = hybrid_search_rag(query, ensemble_retriever, llm)
    
    # Display results
    print("\n📝 FINAL ANSWER:")
    print("-" * 50)
    print(result["final_answer"])
    
    print("\n💡 HYBRID SEARCH EXPLANATION:")
    print("-" * 50)
    print(result["hybrid_search_explanation"])
    
    print("\n📄 RETRIEVED CONTEXT (first 300 characters):")
    print("-" * 50)
    print(result["retrieved_context"][:300] + "...")
    
print("\n" + "=" * 70)
print("✅ ALL QUERIES PROCESSED SUCCESSFULLY!")
print("=" * 70)
print("""
💡 KEY TAKEAWAYS:

1. Hybrid search combines BM25 (keyword) and Vector (semantic) search
2. BM25 excels at finding exact matches for technical terms
3. Vector search understands meaning and finds conceptually related content
4. EnsembleRetriever merges both for comprehensive retrieval
5. Weights can be adjusted based on query types (0.5/0.5 is balanced)
""")